# 15 — Tableau de bord télémétrie Jupyter

Tableau de bord **dans Jupyter** : graphique live sur 15 secondes. Il ne s’agit pas encore d’un HUD superposé à Assetto Corsa et il n’envoie aucune commande.

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaEnv').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvre le depot IA_projet_Ouarti_ds_github dans VS Code : son dossier assetto_corsa_gym doit exister.')
PROJECT=REPO/'OUARTI'
if not PROJECT.is_dir():
    raise FileNotFoundError('Dossier OUARTI absent : extraire OUARTI a la racine du depot.')
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
print('Depot :',REPO)
print('Notebooks :',PROJECT)
print('Python :',sys.version.split()[0],sys.executable)


In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'
PORT_UDP=2345
PORT_TCP=2347

def management(message,timeout=3):
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout);s.sendall(message.encode('ascii'))
        data=b''
        end=time.monotonic()+timeout
        while time.monotonic()<end and len(data)<1024*1024:
            try:chunk=s.recv(65536)
            except socket.timeout:break
            if not chunk:break
            data+=chunk
            try:return json.loads(data)
            except (json.JSONDecodeError, UnicodeDecodeError):pass
            try:return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeDecodeError):pass
        if data:
            raise RuntimeError('Reponse TCP incomplete : '+repr(data[:160]))
    raise RuntimeError('Pas de reponse sur le port management')

def open_telemetry(timeout=4):
    s=socket.socket(socket.AF_INET,socket.SOCK_DGRAM);s.settimeout(.4)
    s.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        s.send(b'connect')
        try:response=s.recv(2**18)
        except socket.timeout:continue
        if response==b'identified':return s
    s.close()
    raise TimeoutError('Pas de confirmation UDP sur 2345 : verifier jeu en piste et plugin sensors_par.')

def next_packet(s,timeout=3):
    end=time.monotonic()+timeout
    while time.monotonic()<end:
        try:message=s.recv(2**18)
        except socket.timeout:continue
        if message==b'disconnect':raise ConnectionError('Le jeu a ferme le flux UDP')
        if message==b'identified':continue
        raw=json.loads(message)
        if not isinstance(raw,dict):continue
        raw['_capture_wall_time']=time.time()
        return raw
    raise TimeoutError('Aucun echantillon recu.')

def close_telemetry(s):
    if s is not None:
        try:s.send(b'disconnect')
        except OSError:pass
        s.close()


In [ ]:
import pandas as pd,matplotlib.pyplot as plt
from IPython.display import display,clear_output
DUREE=15
rows=[];conn=None
try:
    conn=open_telemetry();start=time.monotonic()
    while time.monotonic()-start<DUREE:
        sample=next_packet(conn)
        rows.append({'temps':sample.get('_capture_wall_time'),
                     'vitesse':sample.get('speed'),
                     'volant':sample.get('steerAngle'),
                     'accelerateur':sample.get('accStatus'),
                     'frein':sample.get('brakeStatus')})
        if len(rows)%15==0:
            clear_output(wait=True)
            print('Recu :',len(rows),'echantillons')
            d=pd.DataFrame(rows)
            fig,ax=plt.subplots(figsize=(10,3))
            ax.plot(d.index,d.vitesse,label='vitesse');ax.legend();ax.grid(True)
            display(fig);plt.close(fig)
finally:
    close_telemetry(conn)
print('Fin du diagnostic.')
